# Q1. MLP versus CNN, and memorisation

Purpose: test one reason a plain MLP is weak on CIFAR-10, and separate it from "not enough parameters" and "it overfits". Then watch a wide MLP train on partly randomised labels.

Question: Q1. The models live in `src/common/models.py` (first introduced here). The split is the one saved in Phase 1.

How to run the full experiment, on a GPU node:

`srun --partition=mtech --gres=gpu:1 --cpus-per-task=4 --mem=16G --time=03:00:00 bash scripts/q1_srun.sh`

The rules below were written to `configs/q1_fairness.json` before training. This notebook reads that file. It does not choose a new learning rate.

## What is being compared

The reason under test is missing spatial weight sharing. A convolution reuses one 3×3 filter at every position. A dense layer needs a new weight for every pixel.

That comparison is easy to stack. A matched parameter budget makes the MLP narrow, so a wide MLP (about 8.5× the parameters, same optimiser and epoch budget) asks whether capacity was the real problem. The same narrow MLP with dropout 0.3 and a stronger weight decay asks whether the problem was overfitting. No model in this comparison is augmented, and none uses batch normalisation, so those two knobs cannot explain a gap.

One learning rate is used for every model. It is the winner of an 8-epoch validation probe on the matched MLP and the matched CNN. The wide and regularised MLPs do not get a private search. Test images are scored only after that choice.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.common.models import build_model, count_parameters

plan = json.loads((ROOT / "configs" / "q1_fairness.json").read_text())
print("written_before_training", plan["written_before_training"])
print(plan["reason_under_test"])
print()
for name, spec in plan["models"].items():
    got = count_parameters(build_model(spec))
    print(f"{name:20} planned {spec['parameters']:8d}  built {got:8d}")
    print("   ", spec["description"])

## Expectations written before the run

These are guesses. The report should follow the measured curves if they disagree with the guesses.

In [ ]:
for key, text in plan["expectations_before_training"].items():
    print(key)
    print(text)
    print()

## Results

After `scripts/q1_train.py` has finished, the cells below redraw nothing: they show the files the figure script already wrote from the logs. The reading of those curves is in `EXPLAINER.md`: the matched CNN leads by about 23 test points, a much wider MLP does not close that gap, and at 20% and 50% label noise the validation rise happens while the fit to the random labels is still near chance. The 100% noise run did not fit the random labels.

In [ ]:
from IPython.display import Image, display

table = ROOT / "results" / "q1" / "q1a_model_table.csv"
if not table.exists():
    print("Q1 has not been trained yet.")
else:
    print(table.read_text())
    print((ROOT / "results" / "q1" / "chosen_lr.json").read_text())
    for name in [
        "q1a_lr_probe.png",
        "q1a_val_accuracy.png",
        "q1a_train_accuracy.png",
        "q1a_test_accuracy.png",
        "q1b_memorisation.png",
    ]:
        print(name)
        display(Image(filename=str(ROOT / "figures" / "q1" / name)))